In [ ]:
import pandas as pd

factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

# load just the datasets q01 needs:
def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
import os

def load_orders(data_folder: str, **storage_options) -> pd.DataFrame:
    data_path = os.path.join(data_folder, "orders.tbl")
    # Parse dates during CSV reading to avoid a separate conversion step
    return pd.read_csv(
        data_path,
        sep='|',
        parse_dates=['O_ORDERDATE'],
        infer_datetime_format=True,
        storage_options=storage_options
    )

orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Optimized pandas pipeline using merges to replace isin and map
date = pd.Timestamp("1995-03-04")
cust = customer.loc[customer.C_MKTSEGMENT == "HOUSEHOLD", ["C_CUSTKEY"]]
orders_filt = (
    orders
    .loc[orders.O_ORDERDATE < date, ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_SHIPPRIORITY"]]
    .merge(cust, left_on="O_CUSTKEY", right_on="C_CUSTKEY", how="inner")
    .set_index("O_ORDERKEY")[["O_ORDERDATE", "O_SHIPPRIORITY"]]
)
jn2 = (
    lineitem
    .loc[lineitem.L_SHIPDATE > date, ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]]
    .merge(orders_filt, left_on="L_ORDERKEY", right_index=True, how="inner")
    .assign(TMP=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
)

res = (
    jn2
    .groupby(["L_ORDERKEY", "O_ORDERDATE", "O_SHIPPRIORITY"], as_index=False, sort=False)["TMP"].sum()
    .sort_values("TMP", ascending=False)
    .loc[:, ["L_ORDERKEY", "TMP", "O_ORDERDATE", "O_SHIPPRIORITY"]]
)